In [10]:
import tensorflow as tf

In [11]:
def formula(x, y, b):
    x = tf.matmul(x, y)
    x = x + b
    return x

function_that_uses_a_graph = tf.function(formula)

x1 = tf.constant([[1.0, 2.0]])
y1 = tf.constant([[2.0], [3.0]])
b1 = tf.constant([[4.0]])

orig_value = formula(x1, y1, b1).numpy()
tf_function_value = function_that_uses_a_graph(x1, y1, b1).numpy()

assert(orig_value == tf_function_value)

In [12]:
print(x1.shape)

(1, 2)


In [13]:
print(x1.numpy())

[[1. 2.]]


In [14]:
@tf.function
def formula(x, y, b):
    x = tf.matmul(x, y)
    x = x + b
    return x

function_that_uses_a_graph = tf.function(formula)

x1 = tf.constant([[1.0, 2.0]])
y1 = tf.constant([[2.0], [3.0]])
b1 = tf.constant([[4.0]])

orig_value = formula(x1, y1, b1).numpy()
tf_function_value = function_that_uses_a_graph(x1, y1, b1).numpy()

assert(orig_value == tf_function_value)

In [15]:
@tf.function
def relu_activation(x):
    if tf.greater(x, 0):
        return x
    return 0

print(relu_activation(tf.constant(1)).numpy())
print(relu_activation(tf.constant(-1)).numpy())

1
0


### Автоматичне диференціювання

In [16]:
def f(x):
    return 1 / x ** 2

In [18]:
# похідна функції

x = tf.Variable(2.0)
with tf.GradientTape() as tape:
    y = f(x)
    dydx = tape.gradient(y, x)
    print(dydx)

tf.Tensor(-0.25, shape=(), dtype=float32)


### Створення нейронної мережі

In [19]:
# Примитівна нейронна мережа для функції wx + b

class SimpleModule(tf.Module):
    def __init__(self, name=None):
        super().__init__(name=name)
        self.w = tf.Variable(5.0)
        self.b = tf.Variable(5.0)
    
    def __call__(self, x):
        return self.w * x + self.b


simple_module = SimpleModule(name='simple')
simple_module(tf.constant(5.0))

<tf.Tensor: shape=(), dtype=float32, numpy=30.0>

In [20]:
# Складніша нейронна мережа з двома шарами
class DenseLayer(tf.Module):

    def __init__(self, in_features, out_features, name=None):
        super().__init__(name=name)
        self.w = tf.Variable(
            tf.random.normal([in_features, out_features]), name='w'
        )
        self.b = tf.Variable(tf.zeros([out_features]), name='b')
    
    def __call__(self, x):
        y = tf.matmul(x, self.w) + self.b
        return tf.nn.relu(y)

class NN(tf.Module):

    def __init__(self, name=None):
        super().__init__(name=name)
        self.layer_1 = DenseLayer(in_features=3, out_features=3)
        self.layer_2 = DenseLayer(in_features=3, out_features=1)
    
    def __call__(self, x):
        x = self.layer_1(x)
        return self.layer_2(x)


nn = NN(name='neural_network')
print('Results: ', nn(tf.constant([[2.0, 2.0, 2.0]])))

Results:  tf.Tensor([[0.95442843]], shape=(1, 1), dtype=float32)


### Навчання нейронної мережі

In [21]:
# Створення класу для лінійної моделі

class LinearModel(tf.Module):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.w = tf.Variable(5.0)
        self.b = tf.Variable(0.0)

    def __call__(self, x):
        return self.w * x + self.b

In [33]:
# функція loss обчислює помилку, а функція train - підлаштовує ваги.

def loss(target_y, predicted_y):
    return tf.reduce_mean(tf.square(target_y - predicted_y))

def train(model, x, y, learning_rate):
    with tf.GradientTape() as tape:
        current_loss = loss(y, model(x))
        dw, db = tape.gradient(current_loss, [model.w, model.b])
        model.w.assign_sub(learning_rate * dw)
        model.b.assign_sub(learning_rate * db)

In [34]:
# функція для тренування моделі

def training_loop(model, x, y):
    for epoch in range(10):
        train(model, x, y, learning_rate=0.1)
        current_loss = loss(y, model(x))
        print(f'loss: {current_loss}')

In [35]:
# Тест моделі
TRUE_W = 3.0
TRUE_B = 2.0

NUM_EXAMPLES = 1000

x = tf.random.normal([NUM_EXAMPLES])
noise = tf.random.normal(shape=[NUM_EXAMPLES])
y = x * TRUE_W + TRUE_B + noise

In [36]:
linear_model = LinearModel()
training_loop(linear_model, x, y)

loss: 6.301002025604248
loss: 4.305757999420166
loss: 3.0585484504699707
loss: 2.2786738872528076
loss: 1.7908600568771362
loss: 1.485629916191101
loss: 1.294580101966858
loss: 1.174957036972046
loss: 1.1000317335128784
loss: 1.053086280822754
